In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 0. 导入与数据库连接

In [3]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from psycopg2.extras import execute_values
import utils_z

In [4]:
load_dotenv()
conn = utils_z.get_conn(
    os.getenv("DB_NAME"),
    os.getenv("DB_USER"),
    os.getenv("DB_PASSWORD"),
    os.getenv("DB_HOST"),
    os.getenv("DB_PORT")
)
print("数据库连接成功")

数据库连接成功


In [5]:
lod1_cities = [
    "hamburg", "sanfrancisco",
    "amsterdam", "dresden", "berlin", "rotterdam",
    "luxembourg", "chicago", "boston", "washington",
    "newyork", "portland", "tokyo", "sapporo", "osaka", "kanazawa",
    "toronto", "vienna", "winnipeg", "zurich", "bologna",
    "paris", "lyon", "marseille"
]
print(f"共 {len(lod1_cities)} 个城市")

共 24 个城市


# 1. 建表 — block.lod1_valid_blocks

基础字段 + 已有指标 + 所有待计算指标的预留字段（值均为NULL，后续逐步填入）

In [5]:
utils_z.run_sql("""
    CREATE TABLE IF NOT EXISTS block.lod1_valid_blocks (
        -- 基础字段
        block_id            VARCHAR NOT NULL PRIMARY KEY,
        city                VARCHAR NOT NULL,
        country             VARCHAR NOT NULL,
        geom                GEOMETRY(Polygon, 4326),
        area_m2             NUMERIC,

        -- 已有指标（从各城市表直接复制）
        elongation          NUMERIC,
        compactness         NUMERIC,
        lod1_building_count INTEGER,
        bcr                 NUMERIC,
        
        -- Phase 8: Block自身形态指标
        convexity           NUMERIC,  -- Block凸度
        rectangularity      NUMERIC,   -- Block矩形度

        -- Phase 2: 密度类指标
        far                 NUMERIC,  -- Floor Area Ratio
        bcd                 NUMERIC,  -- Building Coverage Density（栋/公顷）
        osr                 NUMERIC,  -- Open Space Ratio

        -- Phase 3: 高度类指标
        avh                 NUMERIC,  -- 算术平均高度
        avh_w               NUMERIC,  -- 面积加权平均高度
        hsd                 NUMERIC,  -- 高度标准差
        hcv                 NUMERIC,  -- 高度变异系数 = HSD/AVH
        h_max               NUMERIC,  -- 最高建筑高度
        h_range             NUMERIC,  -- 高度极差 = H_max - H_min

        -- Phase 4: 体量类指标
        total_volume        NUMERIC,  -- 总建筑体积
        avg_volume          NUMERIC,  -- 平均建筑体积
        avg_footprint       NUMERIC,  -- 平均建筑底面积
        footprint_cv        NUMERIC,  -- 底面积变异系数

        -- Phase 5: 形态复杂度类指标
        avg_complexity      NUMERIC,  -- 平均建筑复杂度
        avg_rectangularity  NUMERIC,  -- 平均建筑矩形度
        avg_convexity       NUMERIC,  -- 平均建筑凸度

        -- Phase 6: 建筑朝向类指标
        avg_orientation     NUMERIC,  -- 平均方向角（°）
        orientation_entropy NUMERIC,  -- 方向角Shannon熵

        -- Phase 7: 空间分布类指标
        avg_nn_distance     NUMERIC  -- 平均最近邻距离（m）
    );
""", conn=conn)

print("block.lod1_valid_blocks 表创建完成")

# 2. 逐城市导入有效Block + 计算Block自身体态指标 + 建索引

循环24个城市，`WHERE is_valid_lod1 = true`，INSERT汇总表的同时计算：
- `block_convexity` = ST_Area(geom) / ST_Area(ST_ConvexHull(geom))
- `block_rectangularity` = ST_Area(geom) / ST_Area(ST_OrientedEnvelope(geom))

完成后建索引：`(city, block_id)` 联合索引 + `geom` 空间索引

In [9]:
utils_z.run_sql(f"TRUNCATE TABLE block.lod1_valid_blocks CASCADE;", conn=conn)
print(f"lod1_valid_blocks表已清空")

In [ ]:
# 检查汇总表现在是否有数据
result = utils_z.run_sql("SELECT COUNT(*) FROM block.lod1_valid_blocks;", fetch=True, conn=conn)
existing_count = result[0][0]
print(f"汇总表现有记录数: {existing_count}")

if existing_count > 0:
    print("⚠️ 表非空，跳过导入以免重复。如需重建请先手动 TRUNCATE。")
else:
    for city in lod1_cities:
        block_table = f"block.{city}_blocks"
        try:
            sql = f"""
                INSERT INTO block.lod1_valid_blocks
                    (city, country, block_id, geom, area_m2,
                     elongation, compactness, lod1_building_count, bcr,
                     convexity, rectangularity)
                SELECT
                    city, country, block_id, geom, area_m2,
                    elongation, compactness, lod1_building_count, lod1_bcr,
                    -- Phase 8: Block自身体态指标
                    area_m2 / NULLIF(ST_Area(ST_ConvexHull(geom)::geography), 0),
                    area_m2 / NULLIF(ST_Area(ST_OrientedEnvelope(geom)::geography), 0)
                FROM {block_table}
                WHERE is_valid_lod1 = true;
            """
            utils_z.run_sql(sql, conn=conn)
        except Exception as e:
            conn.rollback()
            print(f"  [{city}] ✗ 错误: {e}")

    # Phase 1.3: 建索引
    print("\n开始建索引...")
    utils_z.run_sql("""
        CREATE INDEX IF NOT EXISTS lod1_valid_blocks_city_block_idx
        ON block.lod1_valid_blocks (city, block_id);
    """, conn=conn)
    print("  ✓ city + block_id 联合索引")

    utils_z.run_sql("""
        CREATE INDEX IF NOT EXISTS lod1_valid_blocks_geom_idx
        ON block.lod1_valid_blocks USING GIST (geom);
    """, conn=conn)
    print("  ✓ geom 空间索引")
    print("索引建完")

# 3. 回填原建筑表 floor_count、area、perimeter

对每栋建筑：
- 如果 `floor_count` 有效（非NULL且>0）→ 保留原值
- 否则 → 用 `GREATEST(ROUND(height/3)::integer, 1)` 计算并**回填原表**

这个步骤保证了后续FAR计算时层数信息的完整性。

In [ ]:
for city in lod1_cities:
    bld_table = f"lod1.{city}_buildings_lod1"
    try:
        # 先检查有多少行需要回填
        stats = utils_z.run_sql(f"""
            SELECT
                COUNT(*) AS total,
                COUNT(*) FILTER (WHERE floor_count IS NULL OR floor_count <= 0) AS need_fill
            FROM {bld_table};
        """, fetch=True, conn=conn)[0]

        if stats[1] == 0:
            print(f"  [{city}] 全部有层数信息，跳过")
            continue

        utils_z.run_sql(f"""
            UPDATE {bld_table}
            SET floor_count = GREATEST(ROUND(height / 3)::integer, 1)
            WHERE floor_count IS NULL OR floor_count <= 0;
        """, conn=conn)
        print(f"  [{city}] 回填 {stats[1]}/{stats[0]} 条")
    except Exception as e:
        conn.rollback()
        print(f"  [{city}] ✗ 错误: {e}")

print("\nfloor_count 回填完成")

In [13]:
for city in lod1_cities:
    bld_table = f"lod1.{city}_buildings_lod1"
    try:
        # Step 1: 安全添加字段（已有则跳过）
        with conn.cursor() as cur:
            cur.execute(f"ALTER TABLE {bld_table} ADD COLUMN IF NOT EXISTS area FLOAT;")
            cur.execute(f"ALTER TABLE {bld_table} ADD COLUMN IF NOT EXISTS perimeter FLOAT;")
        conn.commit()

        # Step 2: 统计需要填充的数量
        stats = utils_z.run_sql(f"""
            SELECT
                COUNT(*) AS total,
                COUNT(*) FILTER (WHERE area IS NULL AND geom_2d IS NOT NULL AND ST_IsValid(geom_2d)) AS need_area,
                COUNT(*) FILTER (WHERE perimeter IS NULL AND geom_2d IS NOT NULL AND ST_IsValid(geom_2d)) AS need_perim
            FROM {bld_table};
        """, fetch=True, conn=conn)[0]

        # Step 3: 用 footprint 计算并填充
        if stats[1] > 0:
            utils_z.run_sql(f"""
                UPDATE {bld_table}
                SET area = ST_Area(geom_2d::geography)
                WHERE area IS NULL
                  AND geom_2d IS NOT NULL
                  AND ST_IsValid(geom_2d) = true;
            """, conn=conn)

        if stats[2] > 0:
            utils_z.run_sql(f"""
                UPDATE {bld_table}
                SET perimeter = ST_Perimeter(geom_2d::geography)
                WHERE perimeter IS NULL
                  AND geom_2d IS NOT NULL
                  AND ST_IsValid(geom_2d) = true;
            """, conn=conn)

        print(f"  [{city}] area: {stats[1]}/{stats[0]} 填充, perimeter: {stats[2]}/{stats[0]} 填充")
    except Exception as e:
        conn.rollback()
        print(f"  [{city}] ✗ 错误: {e}")


# 4. Phase 2+3+4: 密度 + 高度 + 体量 — 一次性建筑表聚合

一次扫描建筑表，算出所有依赖建筑数据的指标：
- **密度类**: FAR（容积率）, BCD（栋/公顷）
- **高度类**: AVH, AVH_w, HSD, HCV, H_max, H_range
- **体量类**: total_volume, avg_volume, avg_footprint, footprint_cv

省流：OSR 依赖 BCR 和 FAR，后面单独补算。

In [17]:
for city in lod1_cities:
    bld_table = f"lod1.{city}_buildings_lod1"
    try:
        sql = f"""
            UPDATE block.lod1_valid_blocks sv
            SET
                -- 密度类
                far = CASE WHEN sv.area_m2 > 0
                      THEN agg.total_floor_area / sv.area_m2 END,
                bcd = CASE WHEN sv.area_m2 > 0
                      THEN agg.building_count / (sv.area_m2 / 10000) END,
                -- 高度类
                avh = agg.avg_height,
                avh_w = agg.weighted_avg_height,
                hsd = agg.stddev_height,
                hcv = CASE WHEN agg.avg_height > 0
                      THEN agg.stddev_height / agg.avg_height END,
                h_max = agg.max_height,
                h_range = agg.max_height - agg.min_height,
                -- 体量类
                total_volume = agg.total_volume,
                avg_volume = CASE WHEN agg.building_count > 0
                            THEN agg.total_volume / agg.building_count END,
                avg_footprint = CASE WHEN agg.building_count > 0
                              THEN agg.total_footprint / agg.building_count END,
                footprint_cv = CASE WHEN agg.avg_footprint > 0
                              THEN agg.stddev_footprint / agg.avg_footprint END
            FROM (
                SELECT
                    b.block_id,
                    COUNT(*)                                      AS building_count,
                    SUM(b.area)                                   AS total_footprint,
                    AVG(b.area)                                   AS avg_footprint,
                    STDDEV(b.area)                                AS stddev_footprint,
                    AVG(b.height)                                 AS avg_height,
                    SUM(b.area * b.height)
                        / NULLIF(SUM(b.area), 0)                  AS weighted_avg_height,
                    STDDEV(b.height)                              AS stddev_height,
                    MAX(b.height)                                 AS max_height,
                    MIN(b.height)                                 AS min_height,
                    SUM(b.area * b.floor_count)                   AS total_floor_area,
                    SUM(b.area * b.height)                        AS total_volume
                FROM {bld_table} b
                WHERE b.block_id IS NOT NULL
                GROUP BY b.block_id
            ) agg
            WHERE sv.block_id = agg.block_id;
        """
        utils_z.run_sql(sql, conn=conn)
        print(f"  [{city}] ✓ 密度+高度+体量 计算完成")
    except Exception as e:
        conn.rollback()
        print(f"  [{city}] ✗ 错误: {e}")

print("\nPhase 2+3+4 全部完成")

## 4.1 补充: 计算OSR（开放空间比）

OSR = (1 - BCR) / FAR

依赖 BCR 和 FAR 都算好，单独执行。

In [19]:
utils_z.run_sql("""
    UPDATE block.lod1_valid_blocks
    SET osr = CASE
        WHEN far IS NOT NULL AND far > 0
             AND bcr IS NOT NULL
        THEN (1 - bcr) / far
        ELSE NULL
    END;
""", conn=conn)

cnt = utils_z.run_sql(
    "SELECT COUNT(*) FROM block.lod1_valid_blocks WHERE osr IS NOT NULL;",
    fetch=True, conn=conn
)[0][0]
total = utils_z.run_sql(
    "SELECT COUNT(*) FROM block.lod1_valid_blocks;",
    fetch=True, conn=conn
)[0][0]
print(f"OSR 计算完成: {cnt}/{total} 条有值")

# 5. 形态复杂度类指标

对每栋建筑计算几何复杂度，再聚合到Block级别：
- **avg_complexity**: 周长² / (4π × 底面积)，越接近1越接近圆形
- **avg_rectangularity**: 底面积 / 最小外接矩形面积
- **avg_convexity**: 底面积 / 凸包面积

In [20]:
for city in lod1_cities:
    bld_table = f"lod1.{city}_buildings_lod1"
    try:
        sql = f"""
            UPDATE block.lod1_valid_blocks sv
            SET
                avg_complexity     = agg.avg_complexity,
                avg_rectangularity = agg.avg_rectangularity,
                avg_convexity      = agg.avg_convexity
            FROM (
                SELECT
                    b.block_id,
                    AVG(
                        ST_Perimeter(b.geom_2d) ^ 2
                        / NULLIF(4 * PI() * ST_Area(b.geom_2d), 0)
                    ) AS avg_complexity,
                    AVG(
                        ST_Area(b.geom_2d)
                        / NULLIF(ST_Area(ST_OrientedEnvelope(b.geom_2d)), 0)
                    ) AS avg_rectangularity,
                    AVG(
                        ST_Area(b.geom_2d)
                        / NULLIF(ST_Area(ST_ConvexHull(b.geom_2d)), 0)
                    ) AS avg_convexity
                FROM {bld_table} b
                WHERE b.block_id IS NOT NULL
                GROUP BY b.block_id
            ) agg
            WHERE sv.block_id = agg.block_id;
        """
        utils_z.run_sql(sql, conn=conn)
        print(f"  [{city}] ✓ 形态复杂度 计算完成")
    except Exception as e:
        conn.rollback()
        print(f"  [{city}] ✗ 错误: {e}")

print("\nPhase 5 全部完成")

# 6. 建筑朝向类指标

混合处理流程：
1. **SQL** 计算每栋建筑的最小外接矩形（ST_OrientedEnvelope），导出长轴端点坐标
2. **Python** 计算方向角（0°~180°），聚合 avg_orientation + orientation_entropy
3. **写回** 数据库

In [10]:
def compute_orientation_angle(geom_wkt):
    """
    从建筑几何计算方向角。
    用ST_OrientedEnvelope找出最小外接矩形，取长轴方向，映射到[0°, 180°)。
    """
    from shapely import wkt, LineString
    import math

    poly = wkt.loads(geom_wkt)
    if poly is None or poly.is_empty:
        return None

    # 最小外接矩形
    mbr = poly.minimum_rotated_rectangle
    if mbr is None or mbr.is_empty:
        return None

    coords = list(mbr.exterior.coords)[:-1]  # 去掉闭合点
    if len(coords) < 4:
        return None

    # 计算四条边的长度，找到最长边（长轴）
    edges = []
    for i in range(4):
        x1, y1 = coords[i]
        x2, y2 = coords[(i + 1) % 4]
        length = math.hypot(x2 - x1, y2 - y1)
        edges.append((length, x1, y1, x2, y2))

    # 取最长边
    longest = max(edges, key=lambda e: e[0])
    _, x1, y1, x2, y2 = longest

    # 计算方位角（弧度 → 度），映射到[0°, 180°)
    angle_rad = math.atan2(y2 - y1, x2 - x1)
    angle_deg = math.degrees(angle_rad) % 180.0

    return angle_deg


def shannon_entropy(values, bins=18):
    """
    计算方向角分布的Shannon熵。
    bins: 将0~180°等分，默认每10°一箱。
    """
    if len(values) == 0:
        return None
    hist, _ = np.histogram(values, bins=bins, range=(0, 180))
    probs = hist / hist.sum()
    probs = probs[probs > 0]  # 去掉0概率项
    entropy = -np.sum(probs * np.log(probs))
    return entropy


print('朝向计算函数已定义')

朝向计算函数已定义


In [22]:
for city in lod1_cities:
    bld_table = f"lod1.{city}_buildings_lod1"
    try:
        # Step 1: SQL导出每栋建筑的geom_2d + block_id
        rows = utils_z.run_sql(f"""
            SELECT block_id, ST_AsText(geom_2d)
            FROM {bld_table}
            WHERE block_id IS NOT NULL
              AND geom_2d IS NOT NULL
              AND ST_IsValid(geom_2d) = true;
        """, fetch=True, conn=conn)

        if len(rows) == 0:
            print(f"  [{city}] 无有效建筑几何，跳过")
            continue

        # Step 2: Python计算方向角
        building_angles = []  # [(block_id, angle), ...]
        for block_id, geom_wkt in rows:
            angle = compute_orientation_angle(geom_wkt)
            if angle is not None:
                building_angles.append((block_id, angle))

        # Step 3: 按block聚合
        block_angles = {}
        for block_id, angle in building_angles:
            if block_id not in block_angles:
                block_angles[block_id] = []
            block_angles[block_id].append(angle)

        # Step 4: 计算每个block的 avg_orientation 和 orientation_entropy
        update_data = []
        for block_id, angles in block_angles.items():
            avg_ori = np.mean(angles)
            ent = shannon_entropy(angles, bins=18)
            update_data.append((float(avg_ori), float(ent), block_id))

        # Step 5: 写回汇总表
        with conn.cursor() as cur:
            execute_values(cur, f"""
                UPDATE block.lod1_valid_blocks AS sv
                SET avg_orientation = v.avg_ori,
                    orientation_entropy = v.ent
                FROM (VALUES %s) AS v(avg_ori, ent, block_id)
                WHERE sv.block_id = v.block_id::VARCHAR;
            """, update_data)
        conn.commit()
        print(f"  [{city}] ✓ 朝向计算: {len(update_data)} block / {len(rows)} 建筑")
    except Exception as e:
        conn.rollback()
        print(f"  [{city}] ✗ 错误: {e}")

print("\nPhase 6 全部完成")

# 7. 空间分布类指标 — 平均最近邻距离

对每栋建筑，找同Block内最近的其他建筑的距离，再取Block均值。

注意：
- 只有1栋建筑的Block → avg_nn_distance = NULL
- 空间自连接较重，分城市分批处理

In [ ]:
import metrics_calculator as met_cal

result = met_cal.compute_avg_nn_distance_lod1(conn, cities=lod1_cities, verbose=True)
print(f"\nPhase 7 全部完成 (已计算 {result['computed']}/{result['total_blocks']} 个block)")

共 24 个城市需要处理



NN Distance:   0%|          | 0/24 [00:00<?, ?city/s]e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:121: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"""
NN Distance:   0%|          | 0/24 [00:03<?, ?city/s]

  [hamburg] 建筑: 354270, Block: 5157


NN Distance:   4%|▍         | 1/24 [00:05<02:08,  5.60s/city]e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:121: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"""


  [hamburg] ✓ 更新 5020/5157 个block


...

  [lyon] ✓ 更新 1443/1591 个block


NN Distance:  96%|█████████▌| 23/24 [02:41<00:02,  2.63s/city]

  [marseille] 建筑: 108578, Block: 2536


NN Distance: 100%|██████████| 24/24 [02:41<00:00,  6.74s/city]

  [marseille] ✓ 更新 2396/2536 个block

  验证: avg_nn_distance 计算结果
  Block总数: 208344
  已计算:    204205
  仍为NULL:  4139
  全局均值:  16.93 m
  最小值:    0.0 m
  最大值:    670.69 m

Phase 7 全部完成 (共更新 214299 个block)

Phase 7 全部完成 (已计算 204205/208344 个block)


# 8. 数据验证

按城市检查各字段的完成情况。

In [6]:
metrics = [
    'far', 'bcd', 'osr',
    'avh', 'avh_w', 'hsd', 'hcv', 'h_max', 'h_range',
    'total_volume', 'avg_volume', 'avg_footprint', 'footprint_cv',
    'avg_complexity', 'avg_rectangularity', 'avg_convexity',
    'avg_orientation', 'orientation_entropy',
    'avg_nn_distance',
    'convexity', 'rectangularity'
]

check_cols = ', '.join([
    f"COUNT({m}) AS cnt_{m}" for m in metrics
])

sql = f"""
    SELECT city, COUNT(*) AS total, {check_cols}
    FROM block.lod1_valid_blocks
    GROUP BY city
    ORDER BY city;
"""

df_check = pd.read_sql(sql, conn)
print(f"\n{'='*100}")
print(f"{'城市':<16} {'总数':>6}  " + "  ".join(f"{m:>14}" for m in metrics))
print(f"{'='*100}")
for _, row in df_check.iterrows():
    parts = [f"{row['city']:<16} {int(row['total']):>6}"]
    for m in metrics:
        val = int(row[f'cnt_{m}'])
        parts.append(f"{val:>14}" if val > 0 else f"{'✗':>14}")
    print("  ".join(parts))

# 全局统计
print(f"\n{'='*100}")
print(f"全局总计: {df_check['total'].sum()} 条记录")
for m in metrics:
    total = df_check[f'cnt_{m}'].sum()
    pct = total / df_check['total'].sum() * 100
    print(f"  {m:<22}: {total:>8} 条 ({pct:.1f}%)")

C:\Users\94017\AppData\Local\Temp\ipykernel_15184\3441082709.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_check = pd.read_sql(sql, conn)



城市                   总数             far             bcd             osr             avh           avh_w             hsd             hcv           h_max         h_range    total_volume      avg_volume   avg_footprint    footprint_cv  avg_complexity  avg_rectangularity   avg_convexity  avg_orientation  orientation_entropy  avg_nn_distance       convexity  rectangularity
Amsterdam          2946            2946            2946            2946            2946            2946            2786            2786            2946            2946            2946            2946            2946            2787            2946            2946            2946            2946            2946            2785            2946            2946
Berlin             8726            8726            8726            8726            8726            8726            8617            8617            8726            8726            8726            8726            8726            8617            8726            8726     

In [7]:
metrics = [
    'area_m2', 'elongation', 'compactness', 'lod1_building_count', 'bcr',
    'far', 'bcd', 'osr',
    'avh', 'avh_w', 'hsd', 'hcv', 'h_max', 'h_range',
    'total_volume', 'avg_volume', 'avg_footprint', 'footprint_cv',
    'avg_complexity', 'avg_rectangularity', 'avg_convexity',
    'avg_orientation', 'orientation_entropy',
    'convexity', 'rectangularity'
]

# 分三次查，每次查一种异常
for anomaly_type, condition, label in [
    ("NULL",     "IS NULL", "null"),
    ("零值",     "= 0",     "zero"),
    ("负值",     "< 0",     "neg"),
]:
    filters = ",\n        ".join([
        f"SUM(CASE WHEN {m} {condition} THEN 1 ELSE 0 END) AS {m}"
        for m in metrics
    ])
    sql = f"""
        SELECT city, COUNT(*) AS total, {filters}
        FROM block.lod1_valid_blocks
        GROUP BY city ORDER BY city;
    """
    rows = utils_z.run_sql(sql, fetch=True, conn=conn)
    cities = [r[0] for r in rows]
    totals = [r[1] for r in rows]

    print(f"\n{'='*60}")
    print(f"异常类型: {anomaly_type} ({condition})")
    print(f"{'='*60}")
    print(f"{'指标':<22}", end="")
    for c in cities:
        print(f"  {c[:10]:>10}", end="")
    print()
    print("-" * (22 + 12 * len(cities)))

    for i, m in enumerate(metrics):
        col_idx = 2 + i
        values = [r[col_idx] or 0 for r in rows]
        # 只打印有异常的指标
        if any(v > 0 for v in values):
            print(f"{m:<22}", end="")
            for v, total in zip(values, totals):
                display = f"{v}" if v == 0 else f"⚠{v}"
                print(f"  {display:>10}", end="")
            print()

    # 检查是否所有指标都正常
    all_clean = True
    for i, m in enumerate(metrics):
        col_idx = 2 + i
        if any((r[col_idx] or 0) > 0 for r in rows):
            all_clean = False
            break
    if all_clean:
        print("  所有指标无异常")


异常类型: NULL (IS NULL)
指标                       Amsterdam      Berlin     Bologna      Boston     Chicago     Dresden     Hamburg    Kanazawa  Luxembourg        Lyon   Marseille    New York       Osaka       Paris    Portland   Rotterdam  San Franci     Sapporo       Tokyo     Toronto      Vienna  Washington    Winnipeg      Zurich
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
hsd                           ⚠160        ⚠109         ⚠24        ⚠200        ⚠301         ⚠45        ⚠126        ⚠179         ⚠39        ⚠136        ⚠133        ⚠246        ⚠204        ⚠281        ⚠352        ⚠175        ⚠145        ⚠256        ⚠538        ⚠109         ⚠81        ⚠191         ⚠64         ⚠42
hcv                           ⚠160        ⚠10

## debug: 日本层数、高度有问题

In [17]:
result = utils_z.run_sql("""
    SELECT 
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE hcv IS NULL) AS null_hcv,
        COUNT(*) FILTER (WHERE hsd IS NULL) AS null_hsd,
        COUNT(*) FILTER (WHERE avh = 0) AS zero_avh,
        COUNT(*) FILTER (WHERE avh IS NULL) AS null_avh
    FROM block.lod1_valid_blocks
    WHERE city = 'Tokyo';
""", fetch=True, conn=conn)[0]
print(result)

(55731, 10344, 538, 0, 0)


In [24]:
result = utils_z.run_sql("""
    SELECT 
        block_id, avh, avh_w, hsd, hcv, lod1_building_count
    FROM block.lod1_valid_blocks
    WHERE city = 'Tokyo'
      AND hcv IS NULL
    LIMIT 10;
""", fetch=True, conn=conn)

for row in result:
    print(row)

('JP_TK_088936', Decimal('3.7'), Decimal('3.7'), None, None, 1)
('JP_TK_064285', Decimal('-1803.45909090909'), Decimal('-205.539472698135'), Decimal('4052.0029789487'), None, 11)
('JP_TK_038282', Decimal('-826.85'), Decimal('-48.4951877116993'), Decimal('2888.47836490867'), None, 12)
('JP_TK_073929', Decimal('-151.095238095238'), Decimal('-384.14930110074'), Decimal('1260.73293575584'), None, 63)
('JP_TK_057285', Decimal('-898.881818181818'), Decimal('-5.87203528094755'), Decimal('3018.17933192121'), None, 11)
('JP_TK_041584', Decimal('-1869.1'), Decimal('-84.9074127864047'), Decimal('4033.5604720065'), None, 16)
('JP_TK_038984', Decimal('-384.374509803922'), Decimal('-68.3640397889313'), Decimal('1961.77638734318'), None, 51)
('JP_TK_039444', Decimal('-154.575806451613'), Decimal('0.527467155699966'), Decimal('1270.7396588415'), None, 62)
('JP_TK_045202', Decimal('-425.3'), Decimal('-64.7055518131977'), Decimal('2087.00004203511'), None, 23)
('JP_TK_075736', Decimal('116.5'), Decimal(

In [20]:
result2 = utils_z.run_sql("""
    SELECT 
        block_id, avh, avh_w, hsd, hcv, lod1_building_count
    FROM block.lod1_valid_blocks
    WHERE city = 'Tokyo'
      AND hcv IS NOT NULL
    LIMIT 5;
""", fetch=True, conn=conn)

for row in result2:
    print(row)

('JP_TK_076485', Decimal('24.975'), Decimal('30.5201461726646'), Decimal('13.2793878946713'), Decimal('0.531707223009862'), 8)
('JP_TK_039778', Decimal('8.03846153846154'), Decimal('8.58112507038366'), Decimal('2.06379038887935'), Decimal('0.256739474214656'), 13)
('JP_TK_041748', Decimal('7.225'), Decimal('8.17506939766623'), Decimal('1.9313207915828'), Decimal('0.267310836205231'), 16)
('JP_TK_054383', Decimal('8.41666666666667'), Decimal('8.71666201275327'), Decimal('1.39273004193094'), Decimal('0.165472876269023'), 12)
('JP_TK_055195', Decimal('9.25'), Decimal('10.5640798969332'), Decimal('2.3618495577266'), Decimal('0.255335087321794'), 10)


In [23]:
result = utils_z.run_sql("""
    SELECT COUNT(*)
    FROM block.lod1_valid_blocks
    WHERE city = 'Tokyo'
      AND lod1_building_count = 1;
""", fetch=True, conn=conn)
print(result)

[(538,)]


In [26]:
result = utils_z.run_sql("""
    SELECT 
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE height < 0) AS negative_height,
        MIN(height) AS min_height,
        AVG(height) FILTER (WHERE height < 0) AS avg_negative
    FROM lod1.tokyo_buildings_lod1;
""", fetch=True, conn=conn)[0]
print(f"总建筑: {result[0]}, 负高度: {result[1]}, 最小高度: {result[2]}, 负高度均值: {result[3]}")

总建筑: 2005288, 负高度: 23884, 最小高度: -9999.0, 负高度均值: -9999.0


In [25]:
result = utils_z.run_sql("""
    SELECT 
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE hsd IS NULL) AS null_hsd,
        COUNT(*) FILTER (WHERE hsd = 0) AS zero_hsd,
        COUNT(*) FILTER (WHERE avh <= 0) AS nonpos_avh
    FROM block.lod1_valid_blocks
    WHERE city = 'Tokyo'
      AND hcv IS NULL
      AND lod1_building_count > 1;
""", fetch=True, conn=conn)[0]
print(f"total: {result[0]}, null_hsd: {result[1]}, zero_hsd: {result[2]}, nonpos_avh: {result[3]}")

total: 9806, null_hsd: 0, zero_hsd: 7, nonpos_avh: 9806


In [27]:
japan_cities = ["tokyo", "sapporo", "osaka", "kanazawa"]

for city in japan_cities:
    bld_table = f"lod1.{city}_buildings_lod1"
    result = utils_z.run_sql(f"""
        SELECT
            COUNT(*)                                        AS total,
            ROUND(MAX(height)::numeric, 2)                  AS height_max,
            ROUND(MIN(height)::numeric, 2)                  AS height_min,
            COUNT(*) FILTER (WHERE height IS NULL)          AS height_null,
            COUNT(*) FILTER (WHERE height = 0)              AS height_zero,
            COUNT(*) FILTER (WHERE height < 0)              AS height_negative,
            ROUND(AVG(height)::numeric, 2)                  AS height_avg,
            MAX(floor_count)                                AS floor_max,
            MIN(floor_count)                                AS floor_min,
            COUNT(*) FILTER (WHERE floor_count IS NULL)     AS floor_null,
            COUNT(*) FILTER (WHERE floor_count = 0)         AS floor_zero,
            COUNT(*) FILTER (WHERE floor_count < 0)         AS floor_negative
        FROM {bld_table};
    """, fetch=True, conn=conn)[0]

    print(f"{'='*50}")
    print(f"  {city.upper()} — 建筑原始数据检查")
    print(f"{'='*50}")
    print(f"  总建筑数量:          {result[0]}")
    print(f"  [高度]  最大: {result[1]:>8}  最小: {result[2]:>8}  平均: {result[6]:>8}")
    print(f"           NULL: {result[3]:>6}   为0: {result[4]:>6}   负数: {result[5]:>6}")
    print(f"  [层数]  最大: {result[7]:>8}  最小: {result[8]:>8}")
    print(f"           NULL: {result[9]:>6}   为0: {result[10]:>6}  负数: {result[11]:>6}")
    print()


  TOKYO — 建筑原始数据检查
  总建筑数量:          2005288
  [高度]  最大:   355.50  最小: -9999.00  平均:  -110.09
           NULL:      0   为0:      0   负数:  23884
  [层数]  最大:       86  最小:        1
           NULL:      0   为0:      0  负数:      0

  SAPPORO — 建筑原始数据检查
  总建筑数量:          646474
  [高度]  最大:   173.90  最小: -9999.00  平均:  -238.80
           NULL:      0   为0:      0   负数:  15892
  [层数]  最大:       48  最小:        1
           NULL:      0   为0:      0  负数:      0

  OSAKA — 建筑原始数据检查
  总建筑数量:          616119
  [高度]  最大:   328.40  最小: -9999.00  平均:  -899.38
           NULL:      0   为0:      0   负数:  55935
  [层数]  最大:       75  最小:        1
           NULL:      0   为0:      0  负数:      0

  KANAZAWA — 建筑原始数据检查
  总建筑数量:          213878
  [高度]  最大:    97.20  最小: -9999.00  平均: -1411.21
           NULL:      0   为0:      0   负数:  30331
  [层数]  最大:       30  最小:        1
           NULL:      0   为0:      0  负数:      0



In [28]:
jp_cities = ["tokyo", "sapporo", "osaka", "kanazawa"]

for city in jp_cities:
    bld_table = f"lod1.{city}_buildings_lod1"
    surf_table = f"lod1.{city}_building_surfaces_lod1"
    try:
        # Step 1: 查出高度有问题的建筑及其地面/屋顶Z值
        rows = utils_z.run_sql(f"""
            SELECT
                b.building_id,
                b.height AS old_height,
                gs.z AS ground_z,
                rs.z AS roof_z
            FROM {bld_table} b
            LEFT JOIN (
                SELECT building_id, ST_ZMax(geom_3d) AS z
                FROM {surf_table}
                WHERE surface_type = 'GroundSurface'
            ) gs ON b.building_id = gs.building_id
            LEFT JOIN (
                SELECT building_id, ST_ZMax(geom_3d) AS z
                FROM {surf_table}
                WHERE surface_type = 'RoofSurface'
            ) rs ON b.building_id = rs.building_id
            WHERE (b.height IS NULL OR b.height <= 0)
              AND gs.z IS NOT NULL;
        """, fetch=True, conn=conn)

        # Step 2: 准备更新数据
        update_positive = []  # (new_height, building_id)
        update_null = []      # (building_id,) — 设为NULL
        neg_count = 0

        for row in rows:
            building_id, old_h, ground_z, roof_z = row
            if ground_z is None or roof_z is None:
                update_null.append((building_id,))
                continue
            new_height = float(roof_z) - float(ground_z)
            if new_height > 0:
                update_positive.append((round(new_height, 2), building_id))
            else:
                neg_count += 1
                update_null.append((building_id,))

        # Step 3: 批量更新高度
        if update_positive:
            with conn.cursor() as cur:
                execute_values(cur, f"""
                    UPDATE {bld_table} AS b
                    SET height = v.new_height
                    FROM (VALUES %s) AS v(new_height, building_id)
                    WHERE b.building_id = v.building_id::VARCHAR;
                """, update_positive)
            conn.commit()

        if update_null:
            with conn.cursor() as cur:
                execute_values(cur, f"""
                    UPDATE {bld_table} AS b
                    SET height = NULL
                    FROM (VALUES %s) AS v(building_id)
                    WHERE b.building_id = v.building_id::VARCHAR;
                """, update_null)
            conn.commit()

        print(f"  [{city}] 有问题建筑: {len(rows)}, "
              f"更新为>0: {len(update_positive)}, "
              f"设为NULL: {len(update_null)} "
              f"(其中新计算高度<=0: {neg_count})")

        # Step 4: 对这些建筑的 floor_count 重新计算
        all_ids = [bid for (bid,) in update_null] + [bid for _, bid in update_positive]
        if all_ids:
            with conn.cursor() as cur:
                execute_values(cur, f"""
                    UPDATE {bld_table} AS b
                    SET floor_count = CASE
                        WHEN height IS NOT NULL AND height > 0
                        THEN GREATEST(ROUND(height / 3)::integer, 1)
                        ELSE 0
                    END
                    FROM (VALUES %s) AS v(building_id)
                    WHERE b.building_id = v.building_id::VARCHAR;
                """, [(bid,) for bid in all_ids])
            conn.commit()

    except Exception as e:
        conn.rollback()
        print(f"  [{city}] ✗ 错误: {e}")


  [tokyo] 有问题建筑: 23929, 更新为>0: 23929, 设为NULL: 0 (其中新计算高度<=0: 0)
  [sapporo] 有问题建筑: 15895, 更新为>0: 15895, 设为NULL: 0 (其中新计算高度<=0: 0)
  [osaka] 有问题建筑: 56205, 更新为>0: 56205, 设为NULL: 0 (其中新计算高度<=0: 0)
  [kanazawa] 有问题建筑: 30331, 更新为>0: 30331, 设为NULL: 0 (其中新计算高度<=0: 0)


In [30]:
jp_cities = [
    ("tokyo", "Tokyo"),
    ("sapporo", "Sapporo"),
    ("osaka", "Osaka"),
    ("kanazawa", "Kanazawa"),
]

for city_code, city_name in jp_cities:
    bld_table = f"lod1.{city_code}_buildings_lod1"
    try:
        # Step 1: 直接通过 block_id 匹配，重算高度/层数相关指标
        sql = f"""
            UPDATE block.lod1_valid_blocks sv
            SET
                far = CASE WHEN sv.area_m2 > 0
                      THEN agg.total_floor_area / sv.area_m2 END,
                avh = agg.avg_height,
                avh_w = agg.weighted_avg_height,
                hsd = agg.stddev_height,
                hcv = CASE WHEN agg.avg_height > 0
                      THEN agg.stddev_height / agg.avg_height END,
                h_max = agg.max_height,
                h_range = agg.max_height - agg.min_height,
                total_volume = agg.total_volume,
                avg_volume = CASE WHEN agg.building_count > 0
                            THEN agg.total_volume / agg.building_count END
            FROM (
                SELECT
                    b.block_id,
                    COUNT(*)                                      AS building_count,
                    AVG(b.height)                                 AS avg_height,
                    SUM(b.area * b.height)
                        / NULLIF(SUM(b.area), 0)                  AS weighted_avg_height,
                    STDDEV(b.height)                              AS stddev_height,
                    MAX(b.height)                                 AS max_height,
                    MIN(b.height)                                 AS min_height,
                    SUM(b.area * b.floor_count)                   AS total_floor_area,
                    SUM(b.area * b.height)                        AS total_volume
                FROM {bld_table} b
                WHERE b.block_id IS NOT NULL
                GROUP BY b.block_id
            ) agg
            WHERE sv.block_id = agg.block_id;
        """
        utils_z.run_sql(sql, conn=conn)

        # Step 2: OSR — 使用确切的城市名
        utils_z.run_sql(f"""
            UPDATE block.lod1_valid_blocks
            SET osr = CASE
                WHEN far IS NOT NULL AND far > 0
                     AND bcr IS NOT NULL
                THEN (1 - bcr) / far
                ELSE NULL
            END
            WHERE city = '{city_name}';
        """, conn=conn)

        print(f"  [{city_name}] ✓ 高度/层数相关指标重算完成")
    except Exception as e:
        conn.rollback()
        print(f"  [{city_name}] ✗ 错误: {e}")

print("\n日本4城市指标重算全部完成")


  [Tokyo] ✓ 高度/层数相关指标重算完成
  [Sapporo] ✓ 高度/层数相关指标重算完成
  [Osaka] ✓ 高度/层数相关指标重算完成
  [Kanazawa] ✓ 高度/层数相关指标重算完成

日本4城市指标重算全部完成


## debug: 荷兰数据

In [37]:
for city in ["amsterdam", "rotterdam"]:
    result = utils_z.run_sql(f"""
        SELECT 
            COUNT(*) AS total,
            COUNT(*) FILTER (WHERE height < 0) AS negative_height,
            MIN(height) AS min_height
        FROM lod1.{city}_buildings_lod1;
    """, fetch=True, conn=conn)[0]
    print(f"{city}: 总建筑 {result[0]}, 负高度: {result[1]}, 最小高度: {result[2]}")

amsterdam: 总建筑 165992, 负高度: 248, 最小高度: -2.9260001182556152
rotterdam: 总建筑 189648, 负高度: 21, 最小高度: -4.486000061035156


In [38]:
nl_cities = [
    ("amsterdam", "Amsterdam"),
    ("rotterdam", "Rotterdam"),
]

for city_code, city_name in nl_cities:
    bld_table = f"lod1.{city_code}_buildings_lod1"
    surf_table = f"lod1.{city_code}_building_surfaces_lod1"
    try:
        # ── Step 1: 修复高度 < 2 的建筑 ──
        rows = utils_z.run_sql(f"""
            SELECT
                b.building_id,
                b.height AS old_height,
                gs.z AS ground_z,
                rs.z AS roof_z
            FROM {bld_table} b
            LEFT JOIN (
                SELECT building_id, ST_ZMax(geom_3d) AS z
                FROM {surf_table}
                WHERE surface_type = 'GroundSurface'
            ) gs ON b.building_id = gs.building_id
            LEFT JOIN (
                SELECT building_id, ST_ZMax(geom_3d) AS z
                FROM {surf_table}
                WHERE surface_type = 'RoofSurface'
            ) rs ON b.building_id = rs.building_id
            WHERE (b.height < 2)
              AND gs.z IS NOT NULL;
        """, fetch=True, conn=conn)

        update_positive = []
        update_null = []
        neg_count = 0
        for row in rows:
            building_id, old_h, ground_z, roof_z = row
            if ground_z is None or roof_z is None:
                update_null.append((building_id,))
                continue
            new_h = float(roof_z) - float(ground_z)
            if new_h > 0:
                update_positive.append((round(new_h, 2), building_id))
            else:
                neg_count += 1
                update_null.append((building_id,))

        if update_positive:
            with conn.cursor() as cur:
                execute_values(cur, f"""
                    UPDATE {bld_table} AS b
                    SET height = v.new_height
                    FROM (VALUES %s) AS v(new_height, building_id)
                    WHERE b.building_id = v.building_id::VARCHAR;
                """, update_positive)
            conn.commit()
        if update_null:
            with conn.cursor() as cur:
                execute_values(cur, f"""
                    UPDATE {bld_table} AS b
                    SET height = NULL
                    FROM (VALUES %s) AS v(building_id)
                    WHERE b.building_id = v.building_id::VARCHAR;
                """, update_null)
            conn.commit()
        print(f"  [{city_name}] 高度<2的建筑: {len(rows)}, 更新为>0: {len(update_positive)}, 设为NULL: {len(update_null)}")

        # ── Step 2: 对所有建筑重建 area / perimeter / floor_count ──
        # 2a: area + perimeter
        utils_z.run_sql(f"""
            UPDATE {bld_table}
            SET area = ST_Area(geom_2d::geography),
                perimeter = ST_Perimeter(geom_2d::geography)
            WHERE geom_2d IS NOT NULL AND ST_IsValid(geom_2d) = true;
        """, conn=conn)

        # 2b: floor_count
        utils_z.run_sql(f"""
            UPDATE {bld_table}
            SET floor_count = CASE
                WHEN height IS NOT NULL AND height > 0
                THEN GREATEST(ROUND(height / 3)::integer, 1)
                ELSE 0
            END;
        """, conn=conn)

        print(f"  [{city_name}] area/perimeter/floor_count 全部重建完成")

    except Exception as e:
        conn.rollback()
        print(f"  [{city_name}] ✗ 错误: {e}")

print("\nAmsterdam & Rotterdam 建筑表修复完成")

  [Amsterdam] 高度<2的建筑: 902, 更新为>0: 902, 设为NULL: 0
  [Amsterdam] area/perimeter/floor_count 全部重建完成
  [Rotterdam] 高度<2的建筑: 912, 更新为>0: 912, 设为NULL: 0
  [Rotterdam] area/perimeter/floor_count 全部重建完成

Amsterdam & Rotterdam 建筑表修复完成


In [39]:
nl_cities = [
    ("amsterdam", "Amsterdam"),
    ("rotterdam", "Rotterdam"),
]

for city_code, city_name in nl_cities:
    bld_table = f"lod1.{city_code}_buildings_lod1"
    try:
        # ── Step 1: 重算高度/层数相关指标 ──
        sql = f"""
            UPDATE block.lod1_valid_blocks sv
            SET
                far = CASE WHEN sv.area_m2 > 0
                      THEN agg.total_floor_area / sv.area_m2 END,
                avh = agg.avg_height,
                avh_w = agg.weighted_avg_height,
                hsd = agg.stddev_height,
                hcv = CASE WHEN agg.avg_height > 0
                      THEN agg.stddev_height / agg.avg_height END,
                h_max = agg.max_height,
                h_range = agg.max_height - agg.min_height,
                total_volume = agg.total_volume,
                avg_volume = CASE WHEN agg.building_count > 0
                            THEN agg.total_volume / agg.building_count END
            FROM (
                SELECT
                    b.block_id,
                    COUNT(*)                                      AS building_count,
                    AVG(b.height)                                 AS avg_height,
                    SUM(b.area * b.height)
                        / NULLIF(SUM(b.area), 0)                  AS weighted_avg_height,
                    STDDEV(b.height)                              AS stddev_height,
                    MAX(b.height)                                 AS max_height,
                    MIN(b.height)                                 AS min_height,
                    SUM(b.area * b.floor_count)                   AS total_floor_area,
                    SUM(b.area * b.height)                        AS total_volume
                FROM {bld_table} b
                WHERE b.block_id IS NOT NULL
                GROUP BY b.block_id
            ) agg
            WHERE sv.block_id = agg.block_id;
        """
        utils_z.run_sql(sql, conn=conn)

        # ── Step 2: 重算 OSR（级联依赖） ──
        utils_z.run_sql(f"""
            UPDATE block.lod1_valid_blocks
            SET osr = CASE
                WHEN far IS NOT NULL AND far > 0
                     AND bcr IS NOT NULL
                THEN (1 - bcr) / far
                ELSE NULL
            END
            WHERE city = '{city_name}';
        """, conn=conn)

        print(f"  [{city_name}] ✓ 高度相关指标重算完成")
    except Exception as e:
        conn.rollback()
        print(f"  [{city_name}] ✗ 错误: {e}")

print("\nAmsterdam & Rotterdam 指标重算完成")

  [Amsterdam] ✓ 高度相关指标重算完成
  [Rotterdam] ✓ 高度相关指标重算完成

Amsterdam & Rotterdam 指标重算完成


## debug: 柏林数据

In [9]:
# ── Step 1: 柏林建筑表修复（area / perimeter / floor_count + 高度修正）──
city_code = "berlin"
city_name = "Berlin"
bld_table = f"lod1.{city_code}_buildings_lod1"
surf_table = f"lod1.{city_code}_building_surfaces_lod1"

try:
    # ── 1a: 检查高度异常 ──
    rows = utils_z.run_sql(f"""
        SELECT
            b.building_id,
            b.height AS old_height,
            gs.z AS ground_z,
            rs.z AS roof_z
        FROM {bld_table} b
        LEFT JOIN (
            SELECT building_id, ST_ZMax(geom_3d) AS z
            FROM {surf_table}
            WHERE surface_type = 'GroundSurface'
        ) gs ON b.building_id = gs.building_id
        LEFT JOIN (
            SELECT building_id, ST_ZMax(geom_3d) AS z
            FROM {surf_table}
            WHERE surface_type = 'RoofSurface'
        ) rs ON b.building_id = rs.building_id
        WHERE (b.height IS NULL OR b.height <= 0)
          AND gs.z IS NOT NULL;
    """, fetch=True, conn=conn)

    update_positive = []
    update_null = []
    neg_count = 0
    for row in rows:
        building_id, old_h, ground_z, roof_z = row
        if ground_z is None or roof_z is None:
            update_null.append((building_id,))
            continue
        new_h = float(roof_z) - float(ground_z)
        if new_h > 0:
            update_positive.append((round(new_h, 2), building_id))
        else:
            neg_count += 1
            update_null.append((building_id,))

    if update_positive:
        with conn.cursor() as cur:
            execute_values(cur, f"""
                UPDATE {bld_table} AS b
                SET height = v.new_height
                FROM (VALUES %s) AS v(new_height, building_id)
                WHERE b.building_id = v.building_id::VARCHAR;
            """, update_positive)
        conn.commit()
    if update_null:
        with conn.cursor() as cur:
            execute_values(cur, f"""
                UPDATE {bld_table} AS b
                SET height = NULL
                FROM (VALUES %s) AS v(building_id)
                WHERE b.building_id = v.building_id::VARCHAR;
            """, update_null)
        conn.commit()
    print(f"  [{city_name}] 高度异常建筑: {len(rows)}, 更新为>0: {len(update_positive)}, 设为NULL: {len(update_null)}")

    # ── 1b: 对所有建筑重建 area / perimeter / floor_count ──
    # area + perimeter
    utils_z.run_sql(f"""
        UPDATE {bld_table}
        SET area = ST_Area(geom_2d::geography),
            perimeter = ST_Perimeter(geom_2d::geography)
        WHERE geom_2d IS NOT NULL AND ST_IsValid(geom_2d) = true;
    """, conn=conn)

    # floor_count
    utils_z.run_sql(f"""
        UPDATE {bld_table}
        SET floor_count = CASE
            WHEN height IS NOT NULL AND height > 0
            THEN GREATEST(ROUND(height / 3)::integer, 1)
            ELSE 0
        END;
    """, conn=conn)

    print(f"  [{city_name}] area / perimeter / floor_count 全部重建完成")

except Exception as e:
    conn.rollback()
    print(f"  [{city_name}] ✗ 建筑表修复错误: {e}")


# ── Step 2: Block 全量指标重算 ──

try:
    # 2a: 密度 + 高度 + 体量 + 形态复杂度（一次SQL聚合完成）
    sql = f"""
        UPDATE block.lod1_valid_blocks sv
        SET
            -- 密度类
            far = CASE WHEN sv.area_m2 > 0
                  THEN agg.total_floor_area / sv.area_m2 END,
            bcd = CASE WHEN sv.area_m2 > 0
                  THEN agg.building_count / (sv.area_m2 / 10000) END,
            -- 高度类
            avh = agg.avg_height,
            avh_w = agg.weighted_avg_height,
            hsd = agg.stddev_height,
            hcv = CASE WHEN agg.avg_height > 0
                  THEN agg.stddev_height / agg.avg_height END,
            h_max = agg.max_height,
            h_range = agg.max_height - agg.min_height,
            -- 体量类
            total_volume = agg.total_volume,
            avg_volume = CASE WHEN agg.building_count > 0
                        THEN agg.total_volume / agg.building_count END,
            avg_footprint = CASE WHEN agg.building_count > 0
                          THEN agg.total_footprint / agg.building_count END,
            footprint_cv = CASE WHEN agg.avg_footprint > 0
                          THEN agg.stddev_footprint / agg.avg_footprint END,
            -- 形态复杂度类
            avg_complexity     = agg.avg_complexity,
            avg_rectangularity = agg.avg_rectangularity,
            avg_convexity      = agg.avg_convexity
        FROM (
            SELECT
                b.block_id,
                COUNT(*)                                      AS building_count,
                SUM(b.area)                                   AS total_footprint,
                AVG(b.area)                                   AS avg_footprint,
                STDDEV(b.area)                                AS stddev_footprint,
                AVG(b.height)                                 AS avg_height,
                SUM(b.area * b.height)
                    / NULLIF(SUM(b.area), 0)                  AS weighted_avg_height,
                STDDEV(b.height)                              AS stddev_height,
                MAX(b.height)                                 AS max_height,
                MIN(b.height)                                 AS min_height,
                SUM(b.area * b.floor_count)                   AS total_floor_area,
                SUM(b.area * b.height)                        AS total_volume,
                AVG(
                    ST_Perimeter(b.geom_2d) ^ 2
                    / NULLIF(4 * PI() * ST_Area(b.geom_2d), 0)
                ) AS avg_complexity,
                AVG(
                    ST_Area(b.geom_2d)
                    / NULLIF(ST_Area(ST_OrientedEnvelope(b.geom_2d)), 0)
                ) AS avg_rectangularity,
                AVG(
                    ST_Area(b.geom_2d)
                    / NULLIF(ST_Area(ST_ConvexHull(b.geom_2d)), 0)
                ) AS avg_convexity
            FROM {bld_table} b
            WHERE b.block_id IS NOT NULL
            GROUP BY b.block_id
        ) agg
        WHERE sv.block_id = agg.block_id;
    """
    utils_z.run_sql(sql, conn=conn)
    print(f"  [{city_name}] ✓ 密度+高度+体量+形态复杂度 重算完成")

    # 2b: OSR（级联依赖）
    utils_z.run_sql(f"""
        UPDATE block.lod1_valid_blocks
        SET osr = CASE
            WHEN far IS NOT NULL AND far > 0
                 AND bcr IS NOT NULL
            THEN (1 - bcr) / far
            ELSE NULL
        END
        WHERE city = '{city_name}';
    """, conn=conn)
    print(f"  [{city_name}] ✓ OSR 重算完成")

except Exception as e:
    conn.rollback()
    print(f"  [{city_name}] ✗ SQL指标重算错误: {e}")


# ── Step 3: 朝向类指标重算（Python 处理）──
try:
    rows = utils_z.run_sql(f"""
        SELECT block_id, ST_AsText(geom_2d)
        FROM {bld_table}
        WHERE block_id IS NOT NULL
          AND geom_2d IS NOT NULL
          AND ST_IsValid(geom_2d) = true;
    """, fetch=True, conn=conn)

    if len(rows) > 0:
        building_angles = []
        for block_id, geom_wkt in rows:
            angle = compute_orientation_angle(geom_wkt)
            if angle is not None:
                building_angles.append((block_id, angle))

        block_angles = {}
        for block_id, angle in building_angles:
            if block_id not in block_angles:
                block_angles[block_id] = []
            block_angles[block_id].append(angle)

        update_data = []
        for block_id, angles in block_angles.items():
            avg_ori = np.mean(angles)
            ent = shannon_entropy(angles, bins=18)
            update_data.append((float(avg_ori), float(ent), block_id))

        with conn.cursor() as cur:
            execute_values(cur, f"""
                UPDATE block.lod1_valid_blocks AS sv
                SET avg_orientation = v.avg_ori,
                    orientation_entropy = v.ent
                FROM (VALUES %s) AS v(avg_ori, ent, block_id)
                WHERE sv.block_id = v.block_id::VARCHAR;
            """, update_data)
        conn.commit()
        print(f"  [{city_name}] ✓ 朝向计算: {len(update_data)} blocks / {len(rows)} 建筑")
    else:
        print(f"  [{city_name}] 无有效建筑几何，朝向计算跳过")
except Exception as e:
    conn.rollback()
    print(f"  [{city_name}] ✗ 朝向计算错误: {e}")


# ── Step 4: 空间分布类指标重算 ──
try:
    # 只对 Berlin 这一个城市重算
    import metrics_calculator as met_cal
    result = met_cal.compute_avg_nn_distance_lod1(conn, cities=[city_code], verbose=False)
    print(f"  [{city_name}] ✓ 平均最近邻距离: {result['computed']}/{result['total_blocks']} blocks")
except Exception as e:
    conn.rollback()
    print(f"  [{city_name}] ✗ 最近邻距离错误: {e}")

print(f"\n{city_name} 全量指标重算完成")

  [Berlin] 高度异常建筑: 0, 更新为>0: 0, 设为NULL: 0
  [Berlin] area / perimeter / floor_count 全部重建完成
  [Berlin] ✓ 密度+高度+体量+形态复杂度 重算完成
  [Berlin] ✓ OSR 重算完成
  [Berlin] ✗ 朝向计算错误: name 'compute_orientation_angle' is not defined


e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:120: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"""


  [berlin] 建筑: 844779, Block: 9191
  [berlin] ✓ 更新 8617/9191 个block
  [Berlin] ✓ 平均最近邻距离: 204205/208344 blocks

Berlin 全量指标重算完成


柏林 avg_complexity 重算（过滤退化footprint）

用过滤后的建筑（footprint面积 >= 1 m²）单独重算柏林 avg_complexity。

In [12]:
city_code = "berlin"
city_name = "Berlin"

try:
    # 第一步：重置Berlin的avg_complexity
    utils_z.run_sql(f"""
        UPDATE block.lod1_valid_blocks
        SET avg_complexity = NULL
        WHERE city = '{city_name}';
    """, conn=conn)
    print(f"  [{city_name}] avg_complexity 已重置为 NULL")

    # 第二步：用过滤后的建筑重新计算（过滤掉footprint面积 < 1.0 的退化建筑）
    utils_z.run_sql(f"""
        WITH clean_buildings AS (
            SELECT
                block_id,
                ST_Perimeter(ST_Transform(geom_2d, 3857)) AS perimeter_m,
                ST_Area(ST_Transform(geom_2d, 3857)) AS area_m2
            FROM lod1.{city_code}_buildings_lod1
            WHERE ST_Area(ST_Transform(geom_2d, 3857)) >= 1.0
        ),
        complexity_per_building AS (
            SELECT
                block_id,
                (perimeter_m ^ 2) / NULLIF(area_m2, 0) AS complexity
            FROM clean_buildings
        )
        UPDATE block.lod1_valid_blocks v
        SET avg_complexity = sub.avg_complexity
        FROM (
            SELECT block_id, AVG(complexity) AS avg_complexity
            FROM complexity_per_building
            GROUP BY block_id
        ) sub
        WHERE v.block_id = sub.block_id
          AND v.city = '{city_name}';
    """, conn=conn)

    # 统计结果
    stats = utils_z.run_sql(f"""
        SELECT
            COUNT(*) FILTER (WHERE avg_complexity IS NOT NULL) AS filled,
            COUNT(*) AS total
        FROM block.lod1_valid_blocks
        WHERE city = '{city_name}';
    """, fetch=True, conn=conn)[0]
    print(f"  [{city_name}] avg_complexity 重算完成: {stats[0]}/{stats[1]} blocks")

except Exception as e:
    conn.rollback()
    print(f"  [{city_name}] ✗ 错误: {e}")

  [Berlin] avg_complexity 已重置为 NULL
  [Berlin] avg_complexity 重算完成: 8726/8726 blocks
